In [ ]:
import json, re, math
from pathlib import Path
import pandas as pd
from transformers import AutoTokenizer

# ---------------- settings (fixed; same as Model 1 / Model 3)
MODEL_ID       = "bbarn4/medgemma-27b-text-it-GPTQ"
SERVER_URL     = "http://127.0.0.1:8000/v1"
EVID_DIR       = Path("evidence")
OUT_DIR        = Path("model2_outputs"); OUT_DIR.mkdir(exist_ok=True)
PRED_FILE      = Path("predictions_model2_27b.json")
MAX_NEW_TOKENS, REPETITION_PENALTY = 600, 1.05
CONTEXT_WINDOW, SAFETY, FINAL_MAX  = 40_000, 200, 15
SPECIALISTS = ["Cardiology", "Pulmonology", "Nephrology", "Infectious Diseases"]

tok  = AutoTokenizer.from_pretrained(MODEL_ID)
ntok = lambda s: len(tok.encode(s, add_special_tokens=False))
def prompt_tokens(p):
    return ntok(tok.apply_chat_template([{"role": "user", "content": p}],
                                        tokenize=False, add_generation_prompt=True))

EV = {p.stem: json.load(open(p, encoding="utf-8"))
      for p in sorted(EVID_DIR.glob("*.json")) if p.stem.isdigit()}
assert len(EV) == 60
def case_key(ev): return f"patient_{ev['subject_id']}_{ev['hadm_id']}.txt"

# ---------------- evidence rendering (identical to 7-Shared_Evidence)
def fmt_patient(ev, rows):
    parts, cur = [], None
    for r in sorted(rows):
        c = ev["patient_chunks"][str(r)]
        if c["section"] != cur:
            parts.append(f"\n{c['section']}"); cur = c["section"]
        parts.append(c["text"].strip())
    return "\n".join(parts).strip()

def fmt_kb(ev, groups):
    parts = []
    for header, keys in groups:
        parts.append(f"\n{header}")
        parts += [f"[{ev['kb_chunks'][k]['source_file']}]\n{ev['kb_chunks'][k]['text'].strip()}" for k in keys]
    return "\n\n".join(parts).strip()

# ---------------- KB reduction: k chunks per condition slot, ceil(k/2) from its own guideline
def kb_groups_k(ev, k):
    groups, seen, n_own = [], set(), math.ceil(k / 2)
    for spec in SPECIALISTS:
        for code, c in ev["specialists"][spec]["conditions"].items():
            ranked = sorted(c["kb"], key=lambda x: -x["score"])
            own    = [x["key"] for x in ranked if x["mapped"]][:n_own]
            rest   = [x["key"] for x in ranked if x["key"] not in own][:k - len(own)]
            score  = {x["key"]: x["score"] for x in ranked}
            chosen = sorted(own + rest, key=lambda key: -score[key])
            new = [key for key in chosen if key not in seen]; seen.update(new)
            groups.append((f"{spec} | Candidate condition: {c['name']} ({code})", new))
    return groups

# ---------------- prompt (Model 1 instructions, "record" -> "patient evidence")
MODEL2_INSTRUCTIONS = """
The patient evidence consists of excerpts selected from the clinical record of a single
hospital admission. The guideline evidence consists of excerpts from clinical practice
guidelines; it is reference knowledge only and is not information about this patient.
The candidate conditions in the guideline headings were used only to retrieve evidence;
do not assume the patient has them.

Using only the information in the patient evidence, predict the discharge diagnoses.

List the diagnoses as one list, most important first. Include conditions that were
actively diagnosed or treated during this admission (including new problems and
complications that developed during the stay, as shown by lab trends, medications,
procedures and ICU events) and important chronic conditions.
List each diagnosis once. At most 15 diagnoses.

Use only information supported by the patient evidence.
Do not invent unsupported clinical information.
Output only the numbered list and nothing else.
""".strip()

def model2_prompt(ev, groups):
    return ("PATIENT EVIDENCE:\n" + fmt_patient(ev, ev["model2"]["patient_rows"])
            + "\n\n---\n\nGUIDELINE EVIDENCE:\n" + fmt_kb(ev, groups)
            + "\n\n---\n\nThe patient evidence and guideline evidence are above.\n\n"
            + MODEL2_INSTRUCTIONS)

LIST_RE = re.compile(r"^\s*(?:\d+[.)]|[-*\u2022])\s*(.+?)\s*$")
def parse_diagnoses(raw):
    return [m.group(1) for line in raw.splitlines() if (m := LIST_RE.match(line))]

# ---------------- dry run
log = pd.read_csv("evidence_token_log.csv", dtype={"hadm_id": str}).set_index(["hadm_id", "view"])
mism = sum(ntok(fmt_kb(ev, kb_groups_k(ev, 10))) != log.loc[(h, "Model2")].kb_tokens for h, ev in EV.items())
print("k=10 reproduces the full Model 2 KB view (mismatches, expect 0):", mism)

LIMIT = CONTEXT_WINDOW - MAX_NEW_TOKENS - SAFETY
print(f"Prompt limit: {LIMIT:,} tokens")
for k in range(1, 11):
    t = [prompt_tokens(model2_prompt(ev, kb_groups_k(ev, k))) for ev in EV.values()]
    print(f"k={k:>2}  median={int(pd.Series(t).median()):>6,}  max={max(t):>6,}  fit={sum(x <= LIMIT for x in t)}/60")

k = 2 is the largest value where all 60 cases fit, with a maximum of 36,076 against the 39,200 limit, leaving about 3.1k tokens of headroom. k = 3 already fails 12 cases. The 0 mismatches confirm the evidence is read correctly. At k = 2, each condition gets 1 own-guideline chunk plus the best-scoring remaining chunk, so Model 2 sees at most 40 KB chunks against 200 across the 4 specialists. Put that in the methods.

In [ ]:
# Extra checks: slot count, slot sizes, and whether the 50% floor holds at smaller k
slots  = [c for ev in EV.values() for s in SPECIALISTS for c in ev["specialists"][s]["conditions"].values()]
sizes  = [len(c["kb"]) for c in slots]
mapped = [sum(x["mapped"] for x in c["kb"]) for c in slots]
print("slots per case (expect 20):", {len(kb_groups_k(ev, 1)) for ev in EV.values()})
print("KB chunks per slot: min", min(sizes), "max", max(sizes))
print("own-guideline chunks per slot: min", min(mapped), "max", max(mapped))

def floor_failures(k):
    n_own, bad = math.ceil(k / 2), 0
    for c in slots:
        ranked = sorted(c["kb"], key=lambda x: -x["score"])
        own = [x for x in ranked if x["mapped"]][:n_own]
        bad += len(own) < min(n_own, sum(x["mapped"] for x in ranked))
    return bad
print("floor failures, k=1..10 (expect all 0):", [floor_failures(k) for k in range(1, 11)])

Every slot has exactly 10 chunks, and the own-guideline count runs from 5 to 10. That confirms floor-then-score rather than a fixed 5 + 5. Floor failures are all 0, so the rule is applied correctly.

In [ ]:
# Folder check: which KB folders did each specialist's retrieval draw from?
# The output: one line per specialist, showing each KB folder and how many retrieved chunks came from it across all 60 cases.
from collections import Counter

for spec in SPECIALISTS:
    folders = Counter(ev["kb_chunks"][x["key"]]["folder"]
                      for ev in EV.values()
                      for c in ev["specialists"][spec]["conditions"].values()
                      for x in c["kb"])
    print(f"{spec}: {dict(folders)}")

Cardiology, Pulmonology and Nephrology retrieved only from their own folders. Infectious Diseases also drew 732 of its 3,000 chunks from the pulmonology and nephrology folders. This is because of the shared-family overlap: pneumonia or urinary infections whose guidelines sit in another specialty’s folder.

In [ ]:
# ID chunks by folder x own-guideline flag
idc = Counter((ev["kb_chunks"][x["key"]]["folder"], x["mapped"])
              for ev in EV.values()
              for c in ev["specialists"]["Infectious Diseases"]["conditions"].values()
              for x in c["kb"])
for (folder, mapped), n in sorted(idc.items()): print(f"{folder:22} own={mapped}  {n}")

In [ ]:
# Are ID's other-folder own=False chunks from the shared guideline files?
shared = {ev["kb_chunks"][x["key"]]["source_file"]
          for ev in EV.values()
          for c in ev["specialists"]["Infectious Diseases"]["conditions"].values()
          for x in c["kb"] if x["mapped"] and ev["kb_chunks"][x["key"]]["folder"] != "infectious_diseases"}
other = Counter(ev["kb_chunks"][x["key"]]["source_file"] in shared
                for ev in EV.values()
                for c in ev["specialists"]["Infectious Diseases"]["conditions"].values()
                for x in c["kb"] if not x["mapped"] and ev["kb_chunks"][x["key"]]["folder"] != "infectious_diseases")
print("own=False other-folder chunks from shared guideline files:", dict(other))

Each specialist retrieved only from its own specialty’s guidelines; Infectious Diseases additionally retrieved from the pneumonia and renal-infection guidelines it shares with Pulmonology and Nephrology.

In [ ]:
import requests, os
K = 2
PILOT = ["<hadm_id_1>", "<hadm_id_2>"]   # same 2 cases as the Model 3 pilot
LIMIT = CONTEXT_WINDOW - MAX_NEW_TOKENS - SAFETY

def call_llm(prompt):
    r = requests.post(f"{SERVER_URL}/chat/completions", timeout=1800, json={
        "model": MODEL_ID, "messages": [{"role": "user", "content": prompt}],
        "temperature": 0, "max_tokens": MAX_NEW_TOKENS,
        "repetition_penalty": REPETITION_PENALTY})
    r.raise_for_status(); ch = r.json()["choices"][0]
    return ch["message"]["content"], ch["finish_reason"], r.json()["usage"]

def run_case(hadm):
    out = OUT_DIR / f"{hadm}.json"
    if out.exists(): return json.load(open(out))
    ev = EV[hadm]; prompt = model2_prompt(ev, kb_groups_k(ev, K))
    n = prompt_tokens(prompt); assert n <= LIMIT, (hadm, n)
    raw, finish, usage = call_llm(prompt)
    rec = {"hadm_id": hadm, "case_key": case_key(ev), "k": K, "prompt_tokens": n,
           "finish_reason": finish, "usage": usage, "raw": raw,
           "diagnoses": parse_diagnoses(raw)}
    tmp = out.with_suffix(".tmp"); json.dump(rec, open(tmp, "w"), indent=1); os.replace(tmp, out)
    return rec

for h in PILOT:   # counts only
    r = run_case(h); d = r["diagnoses"]
    print(h, "| prompt", r["prompt_tokens"], "| completion", r["usage"]["completion_tokens"],
          "| finish", r["finish_reason"], "| parsed", len(d), "| unique", len(set(d)))

In [ ]:
# Two cases pilot
# The pilot used the largest and the median-sized case by each system’s own prompt length.

import requests, os
K = 2

M2_TOK = {h: prompt_tokens(model2_prompt(ev, kb_groups_k(ev, K))) for h, ev in EV.items()}
s = pd.Series(M2_TOK)
PILOT = [s.idxmax(), (s - s.median()).abs().idxmin()]
print(PILOT, s[PILOT].tolist())   # expect the max to be 36,076

LIMIT = CONTEXT_WINDOW - MAX_NEW_TOKENS - SAFETY

def call_llm(prompt):
    r = requests.post(f"{SERVER_URL}/chat/completions", timeout=1800, json={
        "model": MODEL_ID, "messages": [{"role": "user", "content": prompt}],
        "temperature": 0, "max_tokens": MAX_NEW_TOKENS,
        "repetition_penalty": REPETITION_PENALTY})
    r.raise_for_status(); ch = r.json()["choices"][0]
    return ch["message"]["content"], ch["finish_reason"], r.json()["usage"]

def run_case(hadm):
    out = OUT_DIR / f"{hadm}.json"
    if out.exists(): return json.load(open(out))
    ev = EV[hadm]; prompt = model2_prompt(ev, kb_groups_k(ev, K))
    n = prompt_tokens(prompt); assert n <= LIMIT, (hadm, n)
    raw, finish, usage = call_llm(prompt)
    rec = {"hadm_id": hadm, "case_key": case_key(ev), "k": K, "prompt_tokens": n,
           "finish_reason": finish, "usage": usage, "raw": raw,
           "diagnoses": parse_diagnoses(raw)}
    tmp = out.with_suffix(".tmp"); json.dump(rec, open(tmp, "w"), indent=1); os.replace(tmp, out)
    return rec

for h in PILOT:   # counts only
    r = run_case(h); d = r["diagnoses"]
    print(h, "| prompt", r["prompt_tokens"], "| completion", r["usage"]["completion_tokens"],
          "| finish", r["finish_reason"], "| parsed", len(d), "| unique", len(set(d)))

In [ ]:
# Full run: It’s 58 calls + the 2 cases kept from the previous cell
done = [run_case(h) for h in sorted(EV)]   # skips the 2 pilot files

fin = Counter(r["finish_reason"] for r in done)
n   = [len(r["diagnoses"]) for r in done]
dup = sum(len(r["diagnoses"]) != len(set(r["diagnoses"])) for r in done)
print("files:", len(done), "| finish:", dict(fin),
      "| parsed min/max:", min(n), max(n), "| cases with duplicates:", dup)

In [ ]:
for r in done:
    d = [x.strip().lower() for x in r["diagnoses"]]
    top = d[:15]
    first_dup = next((i + 1 for i in range(len(d)) if d[i] in d[:i]), None)
    if r["finish_reason"] == "length" or first_dup:
        print(r["hadm_id"], "| finish", r["finish_reason"], "| parsed", len(d),
              "| first duplicate at position", first_dup,
              "| dup in top 15:", len(top) != len(set(top)))

In [ ]:
# Build the predictions file
# The following line loads Model 3’s finished predictions only as a reference, to check two things:
# Keys: Model 2’s file must have the same 60 case keys as Model 3’s.
# Format: each value must have the same type (a list).
m3 = json.load(open("predictions_model3_27b.json"))
pred = {r["case_key"]: r["diagnoses"] for r in done}
assert set(pred) == set(m3), "keys differ from Model 3"
print("Model 3 value type:", type(next(iter(m3.values()))).__name__)   # expect list

# Model 2’s file is written by the following line (json.dump line), to PRED_FILE, which is set to predictions_model2_27b.json in the settings cell.
json.dump(pred, open(PRED_FILE, "w"), indent=1)
print("saved", PRED_FILE, len(pred))

In [ ]:
v = next(iter(m3.values()))
print({key: type(val).__name__ for key, val in v.items()})

In [ ]:
pred = {r["case_key"]: {"raw": r["raw"], "diagnoses": r["diagnoses"],
                        "finish": r["finish_reason"], "prompt_tokens": r["prompt_tokens"]}
        for r in done}
assert set(pred) == set(m3), "keys differ from Model 3"
assert all(set(v) == set(next(iter(m3.values()))) for v in pred.values()), "fields differ"
json.dump(pred, open(PRED_FILE, "w"), indent=1)

chk = json.load(open(PRED_FILE))
print("saved", PRED_FILE, len(chk), "| fields:",
      {key: type(val).__name__ for key, val in next(iter(chk.values())).items()})